# LeetCode #5 — Longest Palindromic Substring: kỹ thuật Expand Around Center

**Mức độ:** Medium · **Chủ đề:** String, Two Pointers, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def longestPalindrome(self, s: str) -> str:
```

Ở bài Palindrome Number, bạn đã biết cách kiểm tra **một** số/chuỗi có đối xứng không. Bài này khó hơn: trong một chuỗi dài, tìm **đoạn đối xứng dài nhất**.

Mục tiêu của notebook là giúp bạn hiểu **kỹ thuật mở rộng từ tâm (Expand Around Center)** — dùng hai con trỏ đi ngược chiều nhau từ giữa ra ngoài.

---

## Sau notebook này, bạn sẽ nắm được

1. Kiểm tra palindrome bằng `s[::-1]` và bằng hai con trỏ.
2. Vì sao thử mọi chuỗi con tốn `O(n³)`.
3. Quan sát then chốt: **mọi palindrome đều có một tâm**, và có **hai loại tâm** (lẻ / chẵn).
4. Cách viết hàm `expand` và tránh lỗi lệch chỉ số khi vòng `while` dừng.
5. Một cách giải khác bằng **Quy hoạch động (DP)** và so sánh hai cách.
6. Mẹo nhận diện bài palindrome trên chuỗi con.
7. Bài tập vận dụng từ dễ đến nâng cao.

# 1. Bài toán là gì?

Cho chuỗi `s`. Hãy trả về **chuỗi con (substring) palindrome dài nhất** trong `s`.

**Palindrome** là chuỗi đọc xuôi hay ngược đều giống nhau: `"aba"`, `"abba"`, `"x"`.

**Constraints (theo đề):**

- `1 <= s.length <= 1000`
- `s` chỉ gồm chữ số và chữ cái tiếng Anh.

---

## Ví dụ 1

```text
s = "babad"  ->  "bab"
```

`"aba"` cũng là đáp án hợp lệ (cùng độ dài 3).

## Ví dụ 2

```text
s = "cbbd"  ->  "bb"
```

---

## Điều quan trọng

1. Trả về **chuỗi**, không phải độ dài.
2. Có thể có **nhiều đáp án** cùng độ dài; LeetCode chấp nhận bất kỳ đáp án nào. Vì vậy khi tự test, ta kiểm tra "có phải palindrome dài nhất không", thay vì so khớp đúng một chuỗi.
3. Chuỗi con phải **liên tiếp** (substring), không phải dãy con (subsequence).

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi đối xứng (palindrome)
2. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi đối xứng (palindrome)

### Định nghĩa và kiểm tra

Chuỗi **palindrome** đọc xuôi hay ngược đều như nhau: `"aba"`, `"abba"`, `"a"`, `""`. Kiểm tra:

- Cách ngắn: `s == s[::-1]` (`O(n)` thời gian và bộ nhớ).
- Hai con trỏ từ hai đầu tiến vào, so từng cặp (`O(1)` bộ nhớ).

In [1]:
def la_doi_xung(s, i=0, j=None):
    j = len(s) - 1 if j is None else j
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True

for w in ["aba", "abba", "abca", "a", ""]:
    print(f"{w!r:>7}: {la_doi_xung(w)} | s == s[::-1]: {w == w[::-1]}")

  'aba': True | s == s[::-1]: True
 'abba': True | s == s[::-1]: True
 'abca': False | s == s[::-1]: False
    'a': True | s == s[::-1]: True
     '': True | s == s[::-1]: True


### Nở từ tâm (expand around center)

Một palindrome đối xứng quanh **tâm**. Tâm là một ký tự (độ dài lẻ, `"aba"`) hoặc **khe giữa** hai ký tự (độ dài chẵn, `"abba"`). Chuỗi dài `n` có `2n − 1` tâm. Từ mỗi tâm, nở hai đầu ra chừng nào hai ký tự còn bằng nhau.

```text
s = "babad", tâm tại 'a' (i = 1):  b[a]b → "bab" ✓ → nở tiếp: hết bên trái → dừng
```

In [2]:
def no_tu_tam(s, l, r):
    while l >= 0 and r < len(s) and s[l] == s[r]:
        l -= 1; r += 1
    return s[l + 1:r]                       # đoạn palindrome dài nhất quanh tâm này

s_pal = "cbbdabadc"
tot_nhat = ""
for i in range(len(s_pal)):
    for l, r in [(i, i), (i, i + 1)]:       # tâm lẻ, tâm chẵn
        p = no_tu_tam(s_pal, l, r)
        if len(p) > len(tot_nhat):
            tot_nhat = p
print("palindrome dài nhất của", repr(s_pal), "là", repr(tot_nhat))

palindrome dài nhất của 'cbbdabadc' là 'dabad'


### Bảng palindrome cho mọi đoạn: `pal[i][j]`

Khi cần hỏi "đoạn `s[i..j]` có đối xứng không?" **nhiều lần**, dựng trước bảng `O(n²)`:

```text
pal[i][j] = (s[i] == s[j]) and (j - i < 2 or pal[i+1][j-1])
```

Ô `(i, j)` cần ô `(i+1, j−1)` → duyệt `i` **từ cuối về đầu**.

In [3]:
def bang_doi_xung(s):
    n = len(s)
    pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            pal[i][j] = s[i] == s[j] and (j - i < 2 or pal[i + 1][j - 1])
    return pal

pal_demo = bang_doi_xung("aab")
for i, row in enumerate(pal_demo):
    print(f"   i={i}:", ["T" if v else "." for v in row])
print("các đoạn đối xứng:", [("aab"[i:j + 1]) for i in range(3) for j in range(i, 3) if pal_demo[i][j]])

   i=0: ['T', 'T', '.']
   i=1: ['.', 'T', '.']
   i=2: ['.', '.', 'T']
các đoạn đối xứng: ['a', 'aa', 'a', 'b']


## 2.2 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [4]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [5]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.3 Áp dụng vào bài này

Bài hỏi **đoạn con đối xứng** dài nhất. Cách chính là **nở từ tâm**: thử cả `2n − 1` tâm (ký tự và khe giữa hai ký tự), mỗi tâm nở ra hai bên — `O(n²)` thời gian, `O(1)` bộ nhớ. Cách thứ hai là **quy hoạch động** trên bảng `pal[i][j]` (đoạn `s[i..j]` đối xứng nếu hai đầu bằng nhau và đoạn bên trong đối xứng) — chính là bảng palindrome ở trên.

# 3. Nhắc lại: kiểm tra palindrome

## Cách 1: so với chuỗi đảo ngược

`t[::-1]` là slicing với bước `-1`, cho ra chuỗi đảo ngược (đã học ở bài Palindrome Number).

In [6]:
for t in ["aba", "abba", "abca", "x"]:
    print(f"{t!r:7} đảo ngược = {t[::-1]!r:7} palindrome? {t == t[::-1]}")

'aba'   đảo ngược = 'aba'   palindrome? True
'abba'  đảo ngược = 'abba'  palindrome? True
'abca'  đảo ngược = 'acba'  palindrome? False
'x'     đảo ngược = 'x'     palindrome? True


## Cách 2: hai con trỏ đi vào giữa

Đặt `i` ở đầu, `j` ở cuối. So `t[i]` với `t[j]`; nếu bằng nhau thì `i += 1`, `j -= 1`. Khác nhau là kết luận ngay "không phải".

```text
a b c b a
i       j     a == a
  i   j       b == b
    ij        gặp nhau -> palindrome
```

Cách này không tạo chuỗi mới và **dừng sớm** khi gặp cặp khác nhau.

In [7]:
def is_palindrome(t: str) -> bool:
    i, j = 0, len(t) - 1
    while i < j:
        if t[i] != t[j]:
            return False
        i += 1
        j -= 1
    return True

print(is_palindrome("abcba"), is_palindrome("abba"), is_palindrome("abca"), is_palindrome(""))

True True False True


# 4. Cách dễ nghĩ nhất: thử mọi chuỗi con

Với mọi cặp `(i, j)`, lấy `s[i:j + 1]`, nếu là palindrome và dài hơn đáp án hiện tại thì cập nhật.

In [8]:
def longest_palindrome_bruteforce(s: str) -> str:
    n = len(s)
    best = ""
    for i in range(n):
        for j in range(i, n):
            t = s[i:j + 1]
            if len(t) > len(best) and is_palindrome(t):
                best = t
    return best

print(longest_palindrome_bruteforce("babad"))
print(longest_palindrome_bruteforce("cbbd"))
print(longest_palindrome_bruteforce("a"))

bab
bb
a


## Độ phức tạp của Brute Force

```text
Số chuỗi con:             ~ n^2 / 2
Kiểm tra mỗi chuỗi con:   O(n)
=> Tổng:                  O(n^3)
```

Với `n = 1000`, `n³ / 2` vào khoảng `5 * 10^8` thao tác — quá chậm với Python.

**Việc thừa ở đâu?** Giả sử đã biết `"bab"` là palindrome. Để kiểm tra `"ababa"`, brute force lại so sánh từ đầu, trong khi ta chỉ cần biết thêm: **hai ký tự mới ở hai đầu có bằng nhau không**.

# 5. Quan sát then chốt: palindrome có "tâm"

Một palindrome đối xứng qua **tâm** của nó. Từ tâm, hai bên lan ra giống hệt nhau.

Có **hai loại tâm**:

**Độ dài lẻ** — tâm là **một ký tự**:

```text
r a c e c a r
      ^
     tâm = 'e'
```

**Độ dài chẵn** — tâm là **khe giữa hai ký tự**:

```text
a b | b a
    ^
   tâm nằm giữa hai chữ 'b'
```

Chuỗi dài `n` có `n` tâm loại lẻ và `n - 1` tâm loại chẵn, tổng cộng `2n - 1` tâm.

> Thay vì hỏi "đoạn `s[i..j]` có phải palindrome không?", ta hỏi "**từ mỗi tâm, palindrome lan ra được xa nhất là bao nhiêu?**"

In [9]:
s = "abba"
print("Tâm lẻ (một ký tự):")
for i in range(len(s)):
    print(f"  tại chỉ số {i}: {s[i]!r}")

print("Tâm chẵn (khe giữa hai ký tự):")
for i in range(len(s) - 1):
    print(f"  giữa {i} và {i + 1}: {s[i]!r} | {s[i + 1]!r}")

Tâm lẻ (một ký tự):
  tại chỉ số 0: 'a'
  tại chỉ số 1: 'b'
  tại chỉ số 2: 'b'
  tại chỉ số 3: 'a'
Tâm chẵn (khe giữa hai ký tự):
  giữa 0 và 1: 'a' | 'b'
  giữa 1 và 2: 'b' | 'b'
  giữa 2 và 3: 'b' | 'a'


# 6. Hàm `expand`: mở rộng từ tâm

Ta đặt hai con trỏ `left`, `right` tại tâm rồi **đi ra ngoài** chừng nào còn:

1. Chưa vượt biên: `left >= 0` và `right < len(s)`.
2. Hai ký tự ở hai đầu bằng nhau: `s[left] == s[right]`.

```python
def expand(s, left, right):
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left -= 1
        right += 1
    return left + 1, right - 1
```

- Tâm lẻ tại `i`: gọi `expand(s, i, i)`.
- Tâm chẵn giữa `i` và `i + 1`: gọi `expand(s, i, i + 1)`.

## Vì sao trả về `left + 1, right - 1`?

Vòng `while` dừng khi bước **vừa thử** đã thất bại (khác nhau hoặc vượt biên). Lúc đó `left`, `right` đã **đi quá** một bước. Palindrome hợp lệ cuối cùng là `s[left + 1 : right]`, tức chỉ số từ `left + 1` tới `right - 1`.

Đây là chỗ người mới hay lệch một đơn vị.

In [10]:
def expand(s: str, left: int, right: int):
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left -= 1
        right += 1
    return left + 1, right - 1


s = "racecar"
lo, hi = expand(s, 3, 3)
print("Tâm lẻ tại 3:", (lo, hi), s[lo:hi + 1])

s = "xabbay"
lo, hi = expand(s, 2, 3)
print("Tâm chẵn giữa 2 và 3:", (lo, hi), s[lo:hi + 1])

s = "abc"
lo, hi = expand(s, 0, 1)   # 'a' != 'b' -> không lan được bước nào
print("Tâm chẵn thất bại:", (lo, hi), repr(s[lo:hi + 1]))

Tâm lẻ tại 3: (0, 6) racecar
Tâm chẵn giữa 2 và 3: (1, 4) abba
Tâm chẵn thất bại: (1, 0) ''


Ở ví dụ cuối, tâm chẵn thất bại ngay: `lo = 1`, `hi = 0`, tức `hi < lo`, độ dài `hi - lo + 1 = 0` — chuỗi rỗng. Không sao cả, nó chỉ không bao giờ trở thành đáp án.

Hãy quan sát `expand` đi từng bước:

In [11]:
def trace_expand(s: str, left: int, right: int):
    print(f"Bắt đầu: left={left}, right={right}")
    while left >= 0 and right < len(s) and s[left] == s[right]:
        print(f"  s[{left}]={s[left]!r} == s[{right}]={s[right]!r} -> palindrome {s[left:right + 1]!r}, lan ra")
        left -= 1
        right += 1
    if left < 0 or right >= len(s):
        print(f"  Dừng: vượt biên (left={left}, right={right})")
    else:
        print(f"  Dừng: s[{left}]={s[left]!r} != s[{right}]={s[right]!r}")
    print(f"  Kết quả: chỉ số {left + 1}..{right - 1} -> {s[left + 1:right]!r}")

trace_expand("babad", 1, 1)

Bắt đầu: left=1, right=1
  s[1]='a' == s[1]='a' -> palindrome 'a', lan ra
  s[0]='b' == s[2]='b' -> palindrome 'bab', lan ra
  Dừng: vượt biên (left=-1, right=3)
  Kết quả: chỉ số 0..2 -> 'bab'


# 7. Lời giải chính

Duyệt mọi `i` từ `0` tới `n - 1`. Với mỗi `i`, thử **cả hai** loại tâm, giữ lại đoạn dài nhất.

Lưu đáp án bằng hai chỉ số `best_lo`, `best_hi` thay vì cắt chuỗi mỗi lần — đỡ tốn bộ nhớ. Chỉ cắt một lần ở cuối.

In [12]:
def longest_palindrome_expand(s: str) -> str:
    best_lo, best_hi = 0, 0

    for i in range(len(s)):
        for lo, hi in (expand(s, i, i), expand(s, i, i + 1)):
            if hi - lo > best_hi - best_lo:
                best_lo, best_hi = lo, hi

    return s[best_lo:best_hi + 1]

print(longest_palindrome_expand("babad"))
print(longest_palindrome_expand("cbbd"))
print(longest_palindrome_expand("a"))
print(longest_palindrome_expand("forgeeksskeegfor"))

bab
bb
a
geeksskeeg


# 8. Mô phỏng từng bước

Xét `s = "cbbd"`.

| i | Loại tâm | Gọi | Lan ra được | Palindrome | Độ dài | best |
|---|---|---|---|---|---|---|
| 0 | lẻ | `expand(0, 0)` | `c` | `"c"` | 1 | `"c"` |
| 0 | chẵn | `expand(0, 1)` | `c ≠ b` | `""` | 0 | `"c"` |
| 1 | lẻ | `expand(1, 1)` | `b`, rồi `c ≠ b` | `"b"` | 1 | `"c"` |
| 1 | chẵn | `expand(1, 2)` | `b == b`, rồi `c ≠ d` | `"bb"` | 2 | **`"bb"`** |
| 2 | lẻ | `expand(2, 2)` | `b`, rồi `b ≠ d` | `"b"` | 1 | `"bb"` |
| 2 | chẵn | `expand(2, 3)` | `b ≠ d` | `""` | 0 | `"bb"` |
| 3 | lẻ | `expand(3, 3)` | `d` | `"d"` | 1 | `"bb"` |
| 3 | chẵn | `expand(3, 4)` | vượt biên | `""` | 0 | `"bb"` |

Nếu **chỉ thử tâm lẻ**, ta sẽ không bao giờ tìm thấy `"bb"` và trả về `"c"` — sai.

In [13]:
def trace_longest_palindrome(s: str) -> str:
    best_lo, best_hi = 0, 0
    for i in range(len(s)):
        for kind, (lo, hi) in (("lẻ  ", expand(s, i, i)), ("chẵn", expand(s, i, i + 1))):
            found = s[lo:hi + 1]
            updated = hi - lo > best_hi - best_lo
            if updated:
                best_lo, best_hi = lo, hi
            mark = "  <- cập nhật best" if updated else ""
            print(f"i={i} tâm {kind}: {found!r:8} độ dài {len(found)}  best={s[best_lo:best_hi + 1]!r}{mark}")
    return s[best_lo:best_hi + 1]

print("Đáp án:", trace_longest_palindrome("cbbd"))

i=0 tâm lẻ  : 'c'      độ dài 1  best='c'
i=0 tâm chẵn: ''       độ dài 0  best='c'
i=1 tâm lẻ  : 'b'      độ dài 1  best='c'
i=1 tâm chẵn: 'bb'     độ dài 2  best='bb'  <- cập nhật best
i=2 tâm lẻ  : 'b'      độ dài 1  best='bb'
i=2 tâm chẵn: ''       độ dài 0  best='bb'
i=3 tâm lẻ  : 'd'      độ dài 1  best='bb'
i=3 tâm chẵn: ''       độ dài 0  best='bb'
Đáp án: bb


# 9. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def longestPalindrome(self, s: str) -> str:
        def expand(left: int, right: int):
            while left >= 0 and right < len(s) and s[left] == s[right]:
                left -= 1
                right += 1
            return left + 1, right - 1

        best_lo, best_hi = 0, 0
        for i in range(len(s)):
            for lo, hi in (expand(i, i), expand(i, i + 1)):
                if hi - lo > best_hi - best_lo:
                    best_lo, best_hi = lo, hi

        return s[best_lo:best_hi + 1]

Vì có thể có nhiều đáp án, hàm `check_answer` kiểm tra 3 điều:

1. Kết quả là **substring** của `s`.
2. Kết quả là **palindrome**.
3. Độ dài bằng **độ dài mong đợi**.

In [15]:
def check_answer(name, s, actual, expected_len):
    assert actual in s, (name, "không phải substring", actual)
    assert actual == actual[::-1], (name, "không phải palindrome", actual)
    assert len(actual) == expected_len, (name, "sai độ dài", actual, expected_len)
    print(f"{name:<28} PASS  {actual!r}")


sol = Solution()
check_answer("Ví dụ 1: babad", "babad", sol.longestPalindrome("babad"), 3)
check_answer("Ví dụ 2: cbbd", "cbbd", sol.longestPalindrome("cbbd"), 2)
check_answer("Một ký tự", "a", sol.longestPalindrome("a"), 1)
check_answer("Hai ký tự khác nhau", "ac", sol.longestPalindrome("ac"), 1)
check_answer("Cả chuỗi là palindrome", "racecar", sol.longestPalindrome("racecar"), 7)
check_answer("Toàn ký tự giống nhau", "aaaa", sol.longestPalindrome("aaaa"), 4)
check_answer("Palindrome chẵn ở giữa", "forgeeksskeegfor", sol.longestPalindrome("forgeeksskeegfor"), 10)
check_answer("Có chữ số", "1221x", sol.longestPalindrome("1221x"), 4)

Ví dụ 1: babad               PASS  'bab'
Ví dụ 2: cbbd                PASS  'bb'
Một ký tự                    PASS  'a'
Hai ký tự khác nhau          PASS  'a'
Cả chuỗi là palindrome       PASS  'racecar'
Toàn ký tự giống nhau        PASS  'aaaa'
Palindrome chẵn ở giữa       PASS  'geeksskeeg'
Có chữ số                    PASS  '1221'


## Đối chiếu với Brute Force

In [16]:
import random

random.seed(5)
for _ in range(1500):
    s = "".join(random.choice("ab1") for _ in range(random.randint(1, 14)))
    fast = sol.longestPalindrome(s)
    slow = longest_palindrome_bruteforce(s)
    assert len(fast) == len(slow), (s, fast, slow)
    assert fast in s and fast == fast[::-1], (s, fast)

print("1500 test ngẫu nhiên: Expand Around Center khớp Brute Force")

1500 test ngẫu nhiên: Expand Around Center khớp Brute Force

# 10. Vì sao thuật toán đúng?

1. **Mọi palindrome đều có đúng một tâm** (một ký tự nếu độ dài lẻ, một khe nếu độ dài chẵn).
2. Gọi `P` là palindrome dài nhất của `s`, tâm của nó là `c`. Khi ta `expand` từ `c`:
   - Mọi bước lan ra bên trong `P` đều thành công vì `P` đối xứng.
   - Vì vậy `expand` lan được **ít nhất** tới hai đầu của `P`.
   - Nó không thể cho ra đoạn **dài hơn** `P`, vì `P` đã dài nhất.
3. Ta thử **mọi** tâm, nên chắc chắn gặp `c`, và đáp án tìm được có độ dài bằng `P`.

# 11. Độ phức tạp

Gọi `n = len(s)`.

- Có `2n - 1` tâm.
- Mỗi lần `expand` lan ra tối đa `O(n)` bước.

```text
Time complexity:  O(n^2)   (trường hợp xấu nhất như "aaaa...a")
Space complexity: O(1)     (chỉ vài biến chỉ số; chuỗi kết quả cắt một lần)
```

Thực tế, với chuỗi "bình thường", hầu hết các tâm dừng sau vài bước, nên chạy nhanh hơn nhiều so với cận trên.

# 12. Cách khác: Quy hoạch động (Dynamic Programming)

Ý tưởng: đoạn `s[i..j]` là palindrome khi và chỉ khi

```text
s[i] == s[j]   và   (đoạn bên trong s[i+1..j-1] là palindrome, hoặc đoạn bên trong có độ dài ≤ 1)
```

Gọi `dp[i][j] = True` nếu `s[i..j]` là palindrome. Vì `dp[i][j]` cần `dp[i+1][j-1]` (đoạn **ngắn hơn**), ta tính theo **độ dài tăng dần**.

Ví dụ: `dp[0][4]` cho `"babab"` cần `s[0] == s[4]` (`b == b`) và `dp[1][3]` (`"aba"` là palindrome) → `True`.

In [17]:
def longest_palindrome_dp(s: str) -> str:
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    best_lo, best_hi = 0, 0

    for length in range(1, n + 1):
        for i in range(0, n - length + 1):
            j = i + length - 1
            if s[i] == s[j] and (length <= 2 or dp[i + 1][j - 1]):
                dp[i][j] = True
                if length > best_hi - best_lo + 1:
                    best_lo, best_hi = i, j

    return s[best_lo:best_hi + 1]

print(longest_palindrome_dp("babad"))
print(longest_palindrome_dp("cbbd"))

for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 12)))
    assert len(longest_palindrome_dp(s)) == len(longest_palindrome_bruteforce(s))
print("DP khớp Brute Force")

bab
bb


DP khớp Brute Force


## So sánh các cách

| Cách làm | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|
| Brute Force | O(n³) | O(1) | Dễ nghĩ nhưng chậm |
| Quy hoạch động | O(n²) | **O(n²)** | Tốn bảng `n × n`; hữu ích nếu cần hỏi nhiều đoạn |
| Expand Around Center | O(n²) | **O(1)** | Gọn, nhanh trong thực tế — nên dùng |

Còn có thuật toán **Manacher** chạy `O(n)`, nhưng phức tạp và hiếm khi cần trong phỏng vấn.

# 13. Những lỗi dễ mắc

## Lỗi 1: Chỉ thử tâm lẻ

In [18]:
def wrong_odd_only(s: str) -> str:
    best_lo, best_hi = 0, 0
    for i in range(len(s)):
        lo, hi = expand(s, i, i)       # thiếu expand(s, i, i + 1)
        if hi - lo > best_hi - best_lo:
            best_lo, best_hi = lo, hi
    return s[best_lo:best_hi + 1]

print("Sai:", repr(wrong_odd_only("cbbd")))
print("Đúng:", repr(longest_palindrome_expand("cbbd")))

Sai: 'c'
Đúng: 'bb'


## Lỗi 2: Quên lùi lại một bước sau `while`

In [19]:
def wrong_expand(s, left, right):
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left -= 1
        right += 1
    return left, right                 # SAI: đã đi quá một bước

lo, hi = wrong_expand("xabay", 2, 2)
print("Sai:", (lo, hi), repr("xabay"[lo:hi + 1]))
lo, hi = expand("xabay", 2, 2)
print("Đúng:", (lo, hi), repr("xabay"[lo:hi + 1]))

Sai: (0, 4) 'xabay'
Đúng: (1, 3) 'aba'


## Lỗi 3: Quên kiểm tra biên trước khi đọc ký tự

Phải viết `left >= 0 and right < len(s)` **trước** `s[left] == s[right]`. Nhờ `and` dừng sớm (short-circuit), Python không đọc `s[right]` khi `right` đã vượt biên.

Đặc biệt nguy hiểm: `s[-1]` **không báo lỗi** trong Python mà trả về ký tự cuối! Nếu thiếu `left >= 0`, ta có thể so nhầm với ký tự cuối chuỗi.

In [20]:
s = "aba"
print(s[-1])   # 'a' — Python cho phép chỉ số âm, nên lỗi này rất khó thấy

a


## Lỗi 4: Lệch chỉ số khi cắt chuỗi

Nếu lưu chỉ số **bao gồm cả hai đầu** `lo..hi` thì cắt bằng `s[lo:hi + 1]`, không phải `s[lo:hi]`.

## Lỗi 5: Trả về độ dài thay vì chuỗi

Đề yêu cầu `-> str`. Trả về `len(...)` sẽ sai kiểu.

# 14. Mẹo nhận diện pattern

Nghĩ tới **Expand Around Center** khi:

- Đề hỏi về **palindrome trên chuỗi con liên tiếp** (dài nhất, đếm số lượng, liệt kê).
- `n` cỡ vài nghìn trở xuống (chấp nhận `O(n²)`).

Mẫu:

```python
for i in range(n):
    xử lý(expand(i, i))       # tâm lẻ
    xử lý(expand(i, i + 1))   # tâm chẵn
```

**Phân biệt:** nếu đề nói **subsequence** (dãy con, được bỏ ký tự), ví dụ **#516 Longest Palindromic Subsequence**, thì expand không dùng được — cần DP khác.

Các bài cùng họ: **#647 Palindromic Substrings** (đếm), **#680 Valid Palindrome II** (hai con trỏ), **#9 Palindrome Number**, **#125 Valid Palindrome**.

# 15. Biến thể: Đếm số chuỗi con palindrome (LeetCode #647)

Mỗi bước lan ra **thành công** của `expand` ứng với **một** palindrome mới. Chỉ cần đếm số bước thành công.

```text
"aaa" -> 6:  "a", "a", "a", "aa", "aa", "aaa"
```

In [21]:
def count_palindromic_substrings(s: str) -> int:
    def count_from(left: int, right: int) -> int:
        count = 0
        while left >= 0 and right < len(s) and s[left] == s[right]:
            count += 1
            left -= 1
            right += 1
        return count

    total = 0
    for i in range(len(s)):
        total += count_from(i, i) + count_from(i, i + 1)
    return total

print(count_palindromic_substrings("abc"))   # 3
print(count_palindromic_substrings("aaa"))   # 6

3
6


# 16. Template tư duy

```python
def expand(left, right):
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left -= 1
        right += 1
    return left + 1, right - 1      # lùi lại bước thất bại

best_lo, best_hi = 0, 0
for i in range(len(s)):
    for lo, hi in (expand(i, i), expand(i, i + 1)):
        if hi - lo > best_hi - best_lo:
            best_lo, best_hi = lo, hi
return s[best_lo:best_hi + 1]
```

Ba điều cần nhớ: **hai loại tâm**, **kiểm tra biên trước**, **trả về `left + 1, right - 1`**.

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `s = "abacab"`:

1. Liệt kê tất cả các tâm (lẻ và chẵn).
2. Với mỗi tâm, palindrome lan ra được là gì?
3. Đáp án cuối cùng là gì?

## Bài 2 — Đoán output

Dùng hàm `expand` ở mục 6:

```python
print(expand("aaaa", 1, 2))
print(expand("abcd", 2, 2))
print(expand("abcd", 3, 4))
```

Gợi ý: với lời gọi cuối, `right = 4` đã vượt biên ngay từ đầu.

## Bài 3 — Kiểm tra palindrome trên một đoạn

Viết hàm kiểm tra `s[lo..hi]` (bao gồm hai đầu) có phải palindrome không, **không tạo chuỗi con mới** (không dùng slicing).

```python
is_pal_range("xabay", 1, 3) -> True    ("aba")
is_pal_range("xabay", 0, 2) -> False   ("xab")
```

In [22]:
def exercise_3_is_pal_range(s: str, lo: int, hi: int) -> bool:
    # TODO:
    # Dùng hai con trỏ lo, hi đi vào giữa
    pass


# assert exercise_3_is_pal_range("xabay", 1, 3) is True
# assert exercise_3_is_pal_range("xabay", 0, 2) is False

## Bài 4 — Điền chỗ trống: lời giải chính

In [23]:
def exercise_4_longest_palindrome(s: str) -> str:
    def expand(left, right):
        # TODO 1: lan ra khi còn trong biên và hai ký tự bằng nhau
        # TODO 2: trả về chỉ số của palindrome hợp lệ cuối cùng
        pass

    best_lo, best_hi = 0, 0
    for i in range(len(s)):
        # TODO 3: thử cả tâm lẻ và tâm chẵn, cập nhật best nếu dài hơn
        pass

    return s[best_lo:best_hi + 1]


# assert len(exercise_4_longest_palindrome("babad")) == 3
# assert exercise_4_longest_palindrome("cbbd") == "bb"

## Bài 5 — Biến thể: độ dài palindrome dài nhất

Viết lại lời giải nhưng chỉ trả về **độ dài**. Hãy để `expand` trả về luôn độ dài thay vì hai chỉ số.

```python
longest_palindrome_length("babad") -> 3
longest_palindrome_length("cbbd")  -> 2
```

In [24]:
def exercise_5_longest_palindrome_length(s: str) -> int:
    # TODO
    pass


# assert exercise_5_longest_palindrome_length("babad") == 3
# assert exercise_5_longest_palindrome_length("cbbd") == 2
# assert exercise_5_longest_palindrome_length("a") == 1

## Bài 6 — Mini challenge: Valid Palindrome II (LeetCode #680)

Cho chuỗi `s`. Trả về `True` nếu `s` có thể trở thành palindrome sau khi **xóa tối đa một ký tự**.

```python
valid_palindrome_ii("aba")   -> True
valid_palindrome_ii("abca")  -> True    (xóa 'b' hoặc 'c')
valid_palindrome_ii("abc")   -> False
```

Yêu cầu: `O(n)` thời gian.

In [25]:
def exercise_6_valid_palindrome_ii(s: str) -> bool:
    # TODO
    pass


# assert exercise_6_valid_palindrome_ii("aba") is True
# assert exercise_6_valid_palindrome_ii("abca") is True
# assert exercise_6_valid_palindrome_ii("abc") is False

# 18. Gợi ý

**Bài 1**
- Mức 1: chuỗi dài 6 → 6 tâm lẻ, 5 tâm chẵn.
- Mức 2: các tâm chẵn ở đây đều thất bại ngay vì không có hai ký tự liền nhau giống nhau.
- Mức 3: hãy thử kỹ tâm lẻ tại chỉ số 3.

**Bài 3**
- Mức 1: giống `is_palindrome` ở mục 3, nhưng bắt đầu từ `lo`, `hi` cho trước.
- Mức 2: `while lo < hi:`.
- Mức 3: khác nhau thì `return False`, ngược lại `lo += 1; hi -= 1`.

**Bài 4**
- Mức 1: so với template mục 16.
- Mức 2: điều kiện `while left >= 0 and right < len(s) and s[left] == s[right]`.
- Mức 3: `return left + 1, right - 1`.

**Bài 5**
- Mức 1: sau vòng `while`, palindrome nằm từ `left + 1` tới `right - 1`.
- Mức 2: độ dài là `(right - 1) - (left + 1) + 1`.
- Mức 3: rút gọn thành `right - left - 1`.

**Bài 6**
- Mức 1: dùng hai con trỏ từ hai đầu như `is_palindrome`.
- Mức 2: lần đầu gặp `s[i] != s[j]`, ta có **hai lựa chọn**: bỏ `s[i]` hoặc bỏ `s[j]`.
- Mức 3: `return is_pal_range(s, i + 1, j) or is_pal_range(s, i, j - 1)`.

# 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

`s = "abacab"`, chỉ số `0..5`.

| Tâm | Palindrome lan được |
|---|---|
| lẻ 0 (`a`) | `a` |
| lẻ 1 (`b`) | `aba` |
| lẻ 2 (`a`) | `a` (vì `b ≠ c`) |
| lẻ 3 (`c`) | **`bacab`** |
| lẻ 4 (`a`) | `a` (vì `c ≠ b`) |
| lẻ 5 (`b`) | `b` |
| mọi tâm chẵn | rỗng |

Đáp án: `"bacab"`.

In [26]:
print(trace_longest_palindrome("abacab"))

i=0 tâm lẻ  : 'a'      độ dài 1  best='a'
i=0 tâm chẵn: ''       độ dài 0  best='a'
i=1 tâm lẻ  : 'aba'    độ dài 3  best='aba'  <- cập nhật best
i=1 tâm chẵn: ''       độ dài 0  best='aba'
i=2 tâm lẻ  : 'a'      độ dài 1  best='aba'
i=2 tâm chẵn: ''       độ dài 0  best='aba'
i=3 tâm lẻ  : 'bacab'  độ dài 5  best='bacab'  <- cập nhật best
i=3 tâm chẵn: ''       độ dài 0  best='bacab'
i=4 tâm lẻ  : 'a'      độ dài 1  best='bacab'
i=4 tâm chẵn: ''       độ dài 0  best='bacab'
i=5 tâm lẻ  : 'b'      độ dài 1  best='bacab'
i=5 tâm chẵn: ''       độ dài 0  best='bacab'
bacab


## Đáp án Bài 2

In [27]:
print(expand("aaaa", 1, 2))   # (0, 3): lan tới hết chuỗi "aaaa"
print(expand("abcd", 2, 2))   # (2, 2): chỉ có "c"
print(expand("abcd", 3, 4))   # (4, 3): rỗng, hi < lo

(0, 3)
(2, 2)
(4, 3)


## Đáp án Bài 3

In [28]:
def answer_3_is_pal_range(s: str, lo: int, hi: int) -> bool:
    while lo < hi:
        if s[lo] != s[hi]:
            return False
        lo += 1
        hi -= 1
    return True

assert answer_3_is_pal_range("xabay", 1, 3) is True
assert answer_3_is_pal_range("xabay", 0, 2) is False
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_longest_palindrome(s: str) -> str:
    def expand(left, right):
        while left >= 0 and right < len(s) and s[left] == s[right]:
            left -= 1
            right += 1
        return left + 1, right - 1

    best_lo, best_hi = 0, 0
    for i in range(len(s)):
        for lo, hi in (expand(i, i), expand(i, i + 1)):
            if hi - lo > best_hi - best_lo:
                best_lo, best_hi = lo, hi
    return s[best_lo:best_hi + 1]

assert len(answer_4_longest_palindrome("babad")) == 3
assert answer_4_longest_palindrome("cbbd") == "bb"
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_longest_palindrome_length(s: str) -> int:
    def expand_len(left, right):
        while left >= 0 and right < len(s) and s[left] == s[right]:
            left -= 1
            right += 1
        return right - left - 1

    best = 0
    for i in range(len(s)):
        best = max(best, expand_len(i, i), expand_len(i, i + 1))
    return best

assert answer_5_longest_palindrome_length("babad") == 3
assert answer_5_longest_palindrome_length("cbbd") == 2
assert answer_5_longest_palindrome_length("a") == 1
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

Khi gặp cặp khác nhau đầu tiên, ta chỉ được xóa một lần, nên thử cả hai cách xóa và kiểm tra phần còn lại.

Vì mỗi lần kiểm tra là `O(n)` và chỉ xảy ra tối đa hai lần, tổng vẫn là `O(n)`.

In [31]:
def answer_6_valid_palindrome_ii(s: str) -> bool:
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return answer_3_is_pal_range(s, i + 1, j) or answer_3_is_pal_range(s, i, j - 1)
        i += 1
        j -= 1
    return True

assert answer_6_valid_palindrome_ii("aba") is True
assert answer_6_valid_palindrome_ii("abca") is True
assert answer_6_valid_palindrome_ii("abc") is False
assert answer_6_valid_palindrome_ii("deeee") is True
print("Bài 6 PASS")

Bài 6 PASS


# 20. Tổng kết

Cốt lõi của bài:

> Mọi palindrome có một **tâm**. Thử từng tâm (cả lẻ lẫn chẵn), dùng hai con trỏ **lan ra ngoài** chừng nào hai đầu còn bằng nhau.

So với bài Palindrome Number (hai con trỏ đi **vào trong** để kiểm tra), ở đây hai con trỏ đi **ra ngoài** để tìm kiếm.

---

## Checklist trước khi nộp

- [ ] Thử **cả** `expand(i, i)` và `expand(i, i + 1)`?
- [ ] Điều kiện biên `left >= 0 and right < len(s)` đứng **trước** phép so sánh ký tự?
- [ ] Sau `while`, trả về `left + 1, right - 1`?
- [ ] Cắt chuỗi đúng `s[lo:hi + 1]`?
- [ ] Trả về **chuỗi**, không phải độ dài?
- [ ] Đã thử `"a"`, `"cbbd"`, `"aaaa"`, `"ac"`?

---

## Khi nào nên nghĩ tới Expand Around Center?

- "palindrome" + "substring" (liên tiếp)
- "dài nhất", "đếm số", "liệt kê" các đoạn đối xứng
- Brute force đang kiểm tra lại các đoạn bên trong nhiều lần